In [1]:
# IMPORT LIBRARY
from pathlib import Path
import polars as pl
import duckdb

# MENENTUKAN FOLDER PROYEK
PROJECT_DIR = Path("/home/jovyan/work")

# LOKASI DATASET MENTAH
RAW_PATH = PROJECT_DIR / "data" / "raw" / "flood_dataset.parquet"

# LOKASI DATASET HASIL CLEANING
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEAN_PATH = PROCESSED_DIR / "flood_dataset_cleaned.parquet"

# MEMASTIKAN FOLDER OUTPUT TERSEDIA
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# MEMERIKSA LOKASI FILE
print("=== PERSIAPAN DATA CLEANING ===")
print("Dataset mentah :", RAW_PATH)
print("File output    :", CLEAN_PATH)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_PATH}"
    )

print("\nDataset mentah berhasil ditemukan.")
print("Folder processed siap digunakan.")

=== PERSIAPAN DATA CLEANING ===
Dataset mentah : /home/jovyan/work/data/raw/flood_dataset.parquet
File output    : /home/jovyan/work/data/processed/flood_dataset_cleaned.parquet

Dataset mentah berhasil ditemukan.
Folder processed siap digunakan.


In [2]:
# MEMBACA DATASET MENTAH
df_raw = pl.read_parquet(RAW_PATH)

print("=== INFORMASI DATASET MENTAH ===")
print(f"Jumlah baris : {df_raw.height:,}")
print(f"Jumlah kolom : {df_raw.width}")

print("\n=== TIPE DATA ===")
print(df_raw.schema)

print("\n=== 5 DATA PERTAMA ===")
print(df_raw.head(5))

=== INFORMASI DATASET MENTAH ===
Jumlah baris : 1,152,432
Jumlah kolom : 13

=== TIPE DATA ===
Schema({'district_id': String, 'date': Datetime(time_unit='us', time_zone=None), 'year': Int32, 'rain_1d': Float64, 'rain_3d': Float64, 'rain_7d': Float64, 'rain_30d': Float64, 'disc_now': Float64, 'disc_7d_mean': Float64, 'disc_change_3d': Float64, 'month': Int32, 'coastal': Int64, 'flood_label': Float64})

=== 5 DATA PERTAMA ===
shape: (5, 13)
┌───────────────┬──────────────┬──────┬─────────┬───┬──────────────┬───────┬─────────┬─────────────┐
│ district_id   ┆ date         ┆ year ┆ rain_1d ┆ … ┆ disc_change_ ┆ month ┆ coastal ┆ flood_label │
│ ---           ┆ ---          ┆ ---  ┆ ---     ┆   ┆ 3d           ┆ ---   ┆ ---     ┆ ---         │
│ str           ┆ datetime[μs] ┆ i32  ┆ f64     ┆   ┆ ---          ┆ i32   ┆ i64     ┆ f64         │
│               ┆              ┆      ┆         ┆   ┆ f64          ┆       ┆         ┆             │
╞═══════════════╪══════════════╪══════╪═════════╪═══

In [3]:
# PEMERIKSAAN VALIDITAS NILAI

print("=== PEMERIKSAAN LABEL BANJIR ===")
print(df_raw["flood_label"].value_counts().sort("flood_label"))

invalid_label = df_raw.filter(
    ~pl.col("flood_label").is_in([0.0, 1.0])
).height

print(f"Label banjir tidak valid: {invalid_label:,}")


print("\n=== PEMERIKSAAN WILAYAH PESISIR ===")
print(df_raw["coastal"].value_counts().sort("coastal"))

invalid_coastal = df_raw.filter(
    ~pl.col("coastal").is_in([0, 1])
).height

print(f"Nilai coastal tidak valid: {invalid_coastal:,}")


print("\n=== PEMERIKSAAN IDENTITAS WILAYAH ===")
district_kosong = df_raw.filter(
    pl.col("district_id").str.strip_chars() == ""
).height

print(f"Identitas wilayah kosong: {district_kosong:,}")


# PEMERIKSAAN NILAI NUMERIK TIDAK TERBATAS
kolom_float = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d",
    "flood_label"
]

print("\n=== PEMERIKSAAN NILAI NON-FINITE ===")

for kolom in kolom_float:
    jumlah = df_raw.filter(
        ~pl.col(kolom).is_finite()
    ).height

    print(f"{kolom}: {jumlah:,}")

=== PEMERIKSAAN LABEL BANJIR ===
shape: (2, 2)
┌─────────────┬─────────┐
│ flood_label ┆ count   │
│ ---         ┆ ---     │
│ f64         ┆ u32     │
╞═════════════╪═════════╡
│ 0.0         ┆ 1068121 │
│ 1.0         ┆ 84311   │
└─────────────┴─────────┘
Label banjir tidak valid: 0

=== PEMERIKSAAN WILAYAH PESISIR ===
shape: (2, 2)
┌─────────┬─────────┐
│ coastal ┆ count   │
│ ---     ┆ ---     │
│ i64     ┆ u32     │
╞═════════╪═════════╡
│ 0       ┆ 115968  │
│ 1       ┆ 1036464 │
└─────────┴─────────┘
Nilai coastal tidak valid: 0

=== PEMERIKSAAN IDENTITAS WILAYAH ===
Identitas wilayah kosong: 0

=== PEMERIKSAAN NILAI NON-FINITE ===
rain_1d: 0
rain_3d: 0
rain_7d: 0
rain_30d: 0
disc_now: 0
disc_7d_mean: 0
disc_change_3d: 0
flood_label: 0


In [4]:
# PROSES DATA CLEANING

kolom_numerik = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d",
    "flood_label"
]

# Membersihkan spasi pada identitas wilayah
df_clean = df_raw.with_columns(
    pl.col("district_id").str.strip_chars()
)

# Menentukan aturan validasi
valid_checks = [
    pl.col("district_id").is_not_null(),
    pl.col("district_id") != "",
    pl.col("date").is_not_null(),
    pl.col("year") == pl.col("date").dt.year(),
    pl.col("month") == pl.col("date").dt.month(),
    pl.col("month").is_between(1, 12),
    pl.col("flood_label").is_in([0.0, 1.0]),
    pl.col("coastal").is_in([0, 1])
]

# Memastikan seluruh nilai numerik finite
valid_checks.extend([
    pl.col(kolom).is_finite()
    for kolom in kolom_numerik
])

# Mempertahankan baris yang memenuhi seluruh aturan
df_clean = df_clean.filter(
    pl.all_horizontal(valid_checks)
)

# Mengurutkan data berdasarkan wilayah dan tanggal
df_clean = df_clean.sort(["district_id", "date"])

# Menyimpan hasil cleaning
df_clean.write_parquet(
    CLEAN_PATH,
    compression="zstd"
)

# Ringkasan hasil cleaning
print("=== HASIL DATA CLEANING ===")
print(f"Baris sebelum cleaning : {df_raw.height:,}")
print(f"Baris setelah cleaning : {df_clean.height:,}")
print(f"Baris yang dikeluarkan : {df_raw.height - df_clean.height:,}")
print(f"Jumlah kolom           : {df_clean.width}")
print(f"Ukuran file hasil      : {CLEAN_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"File berhasil disimpan : {CLEAN_PATH.exists()}")

=== HASIL DATA CLEANING ===
Baris sebelum cleaning : 1,152,432
Baris setelah cleaning : 1,152,432
Baris yang dikeluarkan : 0
Jumlah kolom           : 13
Ukuran file hasil      : 14.53 MB
File berhasil disimpan : True


In [5]:
# VALIDASI DAN PROFILING MENGGUNAKAN DUCKDB

con = duckdb.connect()

CLEAN_PATH_SQL = CLEAN_PATH.as_posix()

# Menghitung jumlah baris hasil cleaning
jumlah_baris_duckdb = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{CLEAN_PATH_SQL}')
""").fetchone()[0]

print("=== VALIDASI DATA CLEANING ===")
print(f"Jumlah baris Polars : {df_clean.height:,}")
print(f"Jumlah baris DuckDB : {jumlah_baris_duckdb:,}")

# DuckDB SUMMARIZE
print("\n=== DUCKDB SUMMARIZE ===")

summary = con.execute(f"""
    SUMMARIZE
    SELECT *
    FROM read_parquet('{CLEAN_PATH_SQL}')
""").df()

print(summary.to_string(index=False))

con.close()

=== VALIDASI DATA CLEANING ===
Jumlah baris Polars : 1,152,432
Jumlah baris DuckDB : 1,152,432

=== DUCKDB SUMMARIZE ===
   column_name column_type                 min                 max  approx_unique                   avg                 std                        q25                        q50                        q75   count  null_percentage
   district_id     VARCHAR      IDN.10.13.10_1         IDN.9.9.9_1            347                   NaN                 NaN                        NaN                        NaN                        NaN 1152432              0.0
          date   TIMESTAMP 2015-01-30 00:00:00 2024-12-31 00:00:00           4543   2020-01-15 12:00:00                 NaN 2017-07-24 05:43:15.719906 2020-01-15 16:49:34.127514 2022-07-09 03:00:27.091217 1152432              0.0
          year     INTEGER                2015                2024             11    2019.5364238410596  2.8556717940748193                       2017                       2020            

In [6]:
# VALIDASI AKHIR DATASET HASIL CLEANING

df_check = pl.read_parquet(CLEAN_PATH)

print("=== VALIDASI FILE HASIL CLEANING ===")

print(f"Jumlah baris       : {df_check.height:,}")
print(f"Jumlah kolom       : {df_check.width}")
print(f"Jumlah duplikat    : {df_check.is_duplicated().sum():,}")

total_missing = sum(df_check.null_count().row(0))
print(f"Total missing value: {total_missing:,}")

print("\n=== VALIDASI KUNCI WILAYAH DAN TANGGAL ===")

duplicate_key = (
    df_check.group_by(["district_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplikasi wilayah-tanggal: {duplicate_key:,}")

# MENYIMPAN HASIL DUCKDB SUMMARIZE
SUMMARY_PATH = PROCESSED_DIR / "duckdb_summarize.csv"

summary.to_csv(SUMMARY_PATH, index=False)

print("\n=== PENYIMPANAN LAPORAN ===")
print(f"File SUMMARIZE tersimpan: {SUMMARY_PATH.exists()}")
print(f"Lokasi file: {SUMMARY_PATH}")

=== VALIDASI FILE HASIL CLEANING ===
Jumlah baris       : 1,152,432
Jumlah kolom       : 13
Jumlah duplikat    : 0
Total missing value: 0

=== VALIDASI KUNCI WILAYAH DAN TANGGAL ===
Duplikasi wilayah-tanggal: 0

=== PENYIMPANAN LAPORAN ===
File SUMMARIZE tersimpan: True
Lokasi file: /home/jovyan/work/data/processed/duckdb_summarize.csv
